In [3]:
import json
import spacy
from typing import List, Tuple, Dict

# Load a spaCy model for tokenization
# You can choose a different model if needed, but this is a standard one.
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model 'en_core_web_sm'. Please wait...")
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def spacy_to_conll(spacy_data: List, output_file: str):
    """
    Converts a list of SpaCy-style annotations to CoNLL format.
    
    Args:
        spacy_data: A list of annotations in the format [text, {"entities": [...]}]
        output_file: The path to the output .txt file.
    """
    with open(output_file, 'w', encoding='utf-8') as f:
        for text, annotations in spacy_data['annotations']:
            doc = nlp(text[:-1])
            
            # Initialize tags list with 'O' for all tokens
            tags = ['O'] * len(doc)
            
            # Get entity spans and convert to BIOES tags
            entities = annotations.get("entities", [])
            for start_char, end_char, label in entities:
                span = doc.char_span(start_char, end_char, label=label)
                if span is None:
                    # Handle cases where character spans don't align with tokens
                    continue
                
                # Apply BIOES tagging scheme
                if len(span) == 1:
                    tags[span.start] = f"S-{label}"
                else:
                    tags[span.start] = f"B-{label}"
                    for i in range(span.start + 1, span.end - 1):
                        tags[i] = f"I-{label}"
                    tags[span.end - 1] = f"E-{label}"
            
            # Write token and tag to file, one per line
            for token, tag in zip(doc, tags):
                f.write(f"{token.text}\t{tag}\n")
            f.write("\n") # Add a blank line to separate sentences

# --- Usage ---
# 1. Make sure your downloaded JSON is in the same directory as this script.
# 2. Replace 'your_spacy_output.json' with the name of your file.
input_json_file = 'annotations.json'
output_conll_file = 'annotated_data.conll'

with open(input_json_file, 'r', encoding='utf-8') as f:
    spacy_annotations = json.load(f)

spacy_to_conll(spacy_annotations, output_conll_file)

print(f"Conversion complete. CoNLL data saved to {output_conll_file}")

Conversion complete. CoNLL data saved to annotated_data.conll
